# Notebook 6: Results Tables
**Paper 5 – Cross-Framework Quantum Algorithm Benchmarking**

Generates all LaTeX-ready tables for the paper:

| Table | Content |
|-------|--------|
| Table 1 | Mean gate count per algorithm per framework |
| Table 2 | Circuit depth comparison |
| Table 3 | Compilation time (mean ± std ms) |
| Table 4 | Simulation time (mean ± std ms, @ 4096 shots) |
| Table 5 | Pairwise JSD with equivalence labels |
| Table 6 | Quantum Volume estimates |
| **Table 7** | **QPack composite scores (NEW)** |

**Prerequisites**: Notebooks 1–4 must be complete.

**Outputs**: `benchmarks/results/tables/*.tex`

In [22]:
import os, sys
QCANVAS_ROOT = os.path.abspath('../..')
if QCANVAS_ROOT not in sys.path:
    sys.path.insert(0, QCANVAS_ROOT)
import pandas as pd
import numpy as np

TABLES_DIR = 'benchmarks/results/tables'
os.makedirs(TABLES_DIR, exist_ok=True)

def save_latex(df, filename, caption, label, float_format='%.3f'):
    """Save a DataFrame as a complete LaTeX table."""
    latex = df.to_latex(float_format=float_format, escape=False,
                        caption=caption, label=label)
    path = os.path.join(TABLES_DIR, filename)
    with open(path, 'w', encoding='utf-8', newline='\n') as f:
        f.write('% Auto-generated by nb06_results_tables.ipynb\n')
        f.write('% Paper 5 - Cross-Framework Quantum Algorithm Benchmarking\n\n')
        f.write(latex)
    print(f'Saved: {path}')

print('LaTeX table generator ready.')

LaTeX table generator ready.


In [23]:
# Load all data
df_struct = pd.read_csv('../metrics/structural_metrics.csv')
df_ctimes = pd.read_csv('../metrics/compilation_times.csv')
df_sim    = pd.read_csv('../metrics/simulation_metrics.csv')
df_qv     = pd.read_csv('../metrics/quantum_volume_estimates.csv')
df_tests  = pd.read_csv('../metrics/statistical_tests.csv')
df_qpack  = pd.read_csv('../metrics/qpack_scores.csv')

print('All metrics CSV files loaded.')

All metrics CSV files loaded.


In [24]:
# ── Table 1: Gate count per algorithm per framework ──────────────────────────
df_fixed = df_struct[df_struct['n_qubits'] == df_struct.groupby('algorithm')['n_qubits'].transform('min')]
t1 = df_fixed.pivot_table(index='algorithm', columns='framework', values='total_gates').round(0).astype(int)
t1.columns.name = 'Framework'
t1 = t1.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 1 — Gate Count:')
display(t1)
save_latex(t1, 'table1_gate_counts.tex',
           caption='Total gate count per algorithm and framework (compiled to OpenQASM 3.0 via QCanvas, minimum qubit count shown).',
           label='tab:gate-counts', float_format='%d')

Table 1 — Gate Count:


Framework,Cirq,PennyLane,Qiskit
algorithm,,,
bell_state,4,4,4
bernstein_vazirani,13,13,13
bit_flip_code,9,10,10
deutsch_jozsa,14,14,14
ghz_state,6,6,6
grovers_algorithm,18,18,18
phase_flip_code,15,16,16
qaoa,12,6,12
qml_xor_classifier,6,6,6


Saved: benchmarks/results/tables\table1_gate_counts.tex


In [25]:
# ── Table 2: Circuit depth ────────────────────────────────────────────────────
t2 = df_fixed.pivot_table(index='algorithm', columns='framework', values='circuit_depth').round(0).astype(int)
t2 = t2.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 2 — Circuit Depth:')
display(t2)
save_latex(t2, 'table2_circuit_depth.tex',
           caption='Estimated circuit depth per algorithm and framework.',
           label='tab:circuit-depth', float_format='%d')

Table 2 — Circuit Depth:


framework,Cirq,PennyLane,Qiskit
algorithm,,,
bell_state,3,3,3
bernstein_vazirani,4,4,4
bit_flip_code,7,8,8
deutsch_jozsa,4,4,4
ghz_state,4,4,4
grovers_algorithm,12,12,12
phase_flip_code,9,10,10
qaoa,9,3,8
qml_xor_classifier,4,4,4


Saved: benchmarks/results/tables\table2_circuit_depth.tex


In [26]:
# ── Table 3: Compilation time (mean ± std) ───────────────────────────────────
df_ct_success = df_ctimes[df_ctimes['success']]
ct_agg = df_ct_success.groupby(['algorithm', 'framework'])[['mean_compile_ms', 'std_compile_ms']].mean()
ct_agg['compile_ms_str'] = ct_agg.apply(
    lambda r: f"{r['mean_compile_ms']:.2f} ± {r['std_compile_ms']:.2f}", axis=1
)
t3 = ct_agg['compile_ms_str'].unstack('framework')
t3 = t3.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 3 — Compilation Time (mean ± std ms):')
display(t3)
save_latex(t3, 'table3_compilation_time.tex',
           caption='Compilation time (mean $\pm$ std, ms, $N=10$ repeats) per algorithm and framework.',
           label='tab:compile-time')

Table 3 — Compilation Time (mean ± std ms):


<>:12: SyntaxWarning: invalid escape sequence '\p'
<>:12: SyntaxWarning: invalid escape sequence '\p'
C:\Users\umerf\AppData\Local\Temp\ipykernel_23596\2095445466.py:12: SyntaxWarning: invalid escape sequence '\p'
  caption='Compilation time (mean $\pm$ std, ms, $N=10$ repeats) per algorithm and framework.',


framework,Cirq,PennyLane,Qiskit
algorithm,,,
bell_state,8.32 ± 0.45,12.18 ± 0.33,18.59 ± 0.40
bernstein_vazirani,12.98 ± 0.62,74.46 ± 1.83,18.07 ± 1.08
bit_flip_code,13.97 ± 1.37,35.04 ± 4.13,55.50 ± 27.01
deutsch_jozsa,12.37 ± 1.02,78.55 ± 2.48,8.06 ± 1.54
ghz_state,10.94 ± 0.63,29.11 ± 1.26,22.56 ± 1.77
grovers_algorithm,32.94 ± 1.90,99.53 ± 5.80,9.85 ± 1.00
phase_flip_code,22.37 ± 5.30,54.68 ± 5.11,64.77 ± 11.44
qaoa,18.89 ± 1.98,53.57 ± 3.80,52.03 ± 1.67
qml_xor_classifier,17.62 ± 1.22,53.60 ± 2.77,47.12 ± 7.92


Saved: benchmarks/results/tables\table3_compilation_time.tex


In [27]:
# ── Table 4: Simulation time (4096 shots) ────────────────────────────────────
df_4k = df_sim[(df_sim['shots'] == 4096) & df_sim['success']]
df_4k_fix = df_4k[df_4k['n_qubits'] == df_4k.groupby('algorithm')['n_qubits'].transform('min')]
sim_agg = df_4k_fix.groupby(['algorithm', 'framework'])[['mean_sim_time_ms', 'std_sim_time_ms']].mean()
sim_agg['sim_ms_str'] = sim_agg.apply(
    lambda r: f"{r['mean_sim_time_ms']:.1f} ± {r['std_sim_time_ms']:.1f}", axis=1
)
t4 = sim_agg['sim_ms_str'].unstack('framework')
t4 = t4.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 4 — Simulation Time (4096 shots, mean ± std ms):')
display(t4)
save_latex(t4, 'table4_simulation_time.tex',
           caption='Simulation time (mean $\pm$ std, ms) at 4096 shots per algorithm and framework. All frameworks simulated on the same QSim Cirq backend.',
           label='tab:sim-time')

Table 4 — Simulation Time (4096 shots, mean ± std ms):


<>:13: SyntaxWarning: invalid escape sequence '\p'
<>:13: SyntaxWarning: invalid escape sequence '\p'
C:\Users\umerf\AppData\Local\Temp\ipykernel_23596\1804536547.py:13: SyntaxWarning: invalid escape sequence '\p'
  caption='Simulation time (mean $\pm$ std, ms) at 4096 shots per algorithm and framework. All frameworks simulated on the same QSim Cirq backend.',


framework,Cirq,PennyLane,Qiskit
algorithm,,,
bell_state,271.2 ± 24.5,274.7 ± 10.1,278.6 ± 15.5
bernstein_vazirani,384.5 ± 14.4,423.9 ± 48.5,386.1 ± 14.0
bit_flip_code,363.6 ± 38.6,349.1 ± 16.6,351.6 ± 18.2
deutsch_jozsa,356.9 ± 37.9,349.8 ± 15.2,361.0 ± 20.0
ghz_state,354.4 ± 29.1,340.9 ± 17.3,342.4 ± 19.4
grovers_algorithm,277.9 ± 8.7,278.4 ± 29.2,270.3 ± 6.0
phase_flip_code,343.7 ± 3.9,362.3 ± 35.7,352.3 ± 8.3
qaoa,274.2 ± 10.0,294.6 ± 14.7,269.2 ± 6.6
qml_xor_classifier,171.9 ± 1.9,171.4 ± 1.4,170.7 ± 1.5


Saved: benchmarks/results/tables\table4_simulation_time.tex


In [28]:
# ── Table 5: Pairwise JSD with labels ────────────────────────────────────────
t5_data = df_tests[['algorithm', 'n_qubits', 'jsd_qiskit_cirq', 'jsd_qiskit_pl',
                      'jsd_cirq_pl', 'label_qk_cq', 'label_qk_pl', 'label_cq_pl']].copy()

# Combine JSD value and label
for pair, label_col in [('jsd_qiskit_cirq', 'label_qk_cq'),
                          ('jsd_qiskit_pl',   'label_qk_pl'),
                          ('jsd_cirq_pl',     'label_cq_pl')]:
    col_name = pair.replace('jsd_', '').replace('_', '↔')
    t5_data[col_name] = t5_data.apply(
        lambda r: f"{r[pair]:.4f} {r[label_col]}" if pd.notna(r[pair]) else '-', axis=1
    )

t5 = t5_data.set_index('algorithm')[['qiskit↔cirq', 'qiskit↔pl', 'cirq↔pl']]
t5.columns = ['Qiskit↔Cirq', 'Qiskit↔PennyLane', 'Cirq↔PennyLane']
print('Table 5 — Pairwise JSD (✓ < 0.01, ~ < 0.05, ✗ ≥ 0.05):')
display(t5)
save_latex(t5, 'table5_pairwise_jsd.tex',
           caption='Pairwise Jensen-Shannon Divergence (JSD, $\sqrt{}$ form) across all algorithms at 4096 shots. \\checkmark: JSD $<$ 0.01 (equivalent); $\\sim$: marginal; \\xmark: JSD $\\geq$ 0.05 (divergent).',
           label='tab:jsd')

Table 5 — Pairwise JSD (✓ < 0.01, ~ < 0.05, ✗ ≥ 0.05):


<>:19: SyntaxWarning: invalid escape sequence '\s'
<>:19: SyntaxWarning: invalid escape sequence '\s'
C:\Users\umerf\AppData\Local\Temp\ipykernel_23596\1357082855.py:19: SyntaxWarning: invalid escape sequence '\s'
  caption='Pairwise Jensen-Shannon Divergence (JSD, $\sqrt{}$ form) across all algorithms at 4096 shots. \\checkmark: JSD $<$ 0.01 (equivalent); $\\sim$: marginal; \\xmark: JSD $\\geq$ 0.05 (divergent).',


,Qiskit↔Cirq,Qiskit↔PennyLane,Cirq↔PennyLane
algorithm,,,
bell_state,0.0035 ✓,0.0002 ✓,0.0037 ✓
bernstein_vazirani,0.0000 ✓,0.0000 ✓,0.0000 ✓
bernstein_vazirani,0.0000 ✓,1.0000 ✗,1.0000 ✗
bernstein_vazirani,0.0000 ✓,1.0000 ✗,1.0000 ✗
bernstein_vazirani,0.0000 ✓,1.0000 ✗,1.0000 ✗
bernstein_vazirani,0.0000 ✓,1.0000 ✗,1.0000 ✗
bernstein_vazirani,0.0000 ✓,1.0000 ✗,1.0000 ✗
bit_flip_code,0.7066 ✗,0.0004 ✓,0.7065 ✗
deutsch_jozsa,0.0000 ✓,0.0000 ✓,0.0000 ✓


Saved: benchmarks/results/tables\table5_pairwise_jsd.tex


In [29]:
# ── Table 6: Quantum Volume estimates ────────────────────────────────────────
df_qv_fix = df_qv[df_qv['n_qubits'] == df_qv.groupby('algorithm')['n_qubits'].transform('min')]
t6 = df_qv_fix.pivot_table(index='algorithm', columns='framework',
                              values='effective_qv').round(0).astype(int)
t6 = t6.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 6 — Quantum Volume Estimates:')
display(t6)
save_latex(t6, 'table6_quantum_volume.tex',
           caption='Estimated Quantum Volume per algorithm and framework. Note: estimated values based on depolarizing + T1 noise model; not measured on real hardware.',
           label='tab:quantum-volume', float_format='%d')

Table 6 — Quantum Volume Estimates:


framework,Cirq,PennyLane,Qiskit
algorithm,,,
bell_state,2048,2048,2048
bernstein_vazirani,2048,2048,2048
bit_flip_code,2048,2048,2048
deutsch_jozsa,2048,2048,2048
ghz_state,2048,2048,2048
grovers_algorithm,2048,2048,2048
phase_flip_code,2048,2048,2048
qaoa,2048,2048,2048
qml_xor_classifier,2048,2048,2048


Saved: benchmarks/results/tables\table6_quantum_volume.tex


In [30]:
# ── Table 7: QPack Composite Scores (NEW) ────────────────────────────────────
if not df_qpack.empty:
    df_qpack_display = df_qpack.set_index('framework').rename(
        index={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'}
    ).rename(columns={
        'S_compile_speed': 'Speed Score',
        'S_scalability':   'Scalability Score',
        'S_accuracy':      'Accuracy Score',
        'S_capacity':      'Capacity Score',
        'S_overall':       'Overall Score',
    })
    print('Table 7 — QPack Composite Scores:')
    display(df_qpack_display.round(2))

    # Add QPack reference row
    ref_note = "QPack ref (QuEST=183.2, Cirq=157.6, Qiskit Aer=147.2)."

    save_latex(df_qpack_display.round(2), 'table7_qpack_scores.tex',
               caption=f'QPack-adapted composite benchmark scores per framework. {ref_note}',
               label='tab:qpack-scores')
else:
    print('[SKIP] qpack_scores.csv not found — run nb03 first.')

Table 7 — QPack Composite Scores:


,Speed Score,Scalability Score,Accuracy Score,Capacity Score,Overall Score
framework,,,,,
Qiskit,2.84,3.91,9.41,8,58.80
Cirq,2.84,3.91,9.41,8,58.79
PennyLane,2.56,-0.90,9.18,2,9.28


Saved: benchmarks/results/tables\table7_qpack_scores.tex
